# Laplace's law on a T4, under the central-moment fluid default

`GPU/src/bubble.cu` names no fluid collision operator, so it takes the library
default in `phasefield.cuh`. That default changed from `MultiOp::BGK` to
`MultiOp::CentralMoments` on 2026-09-18, which makes `bubble.cu` the only driver
in `GPU/` that both inherits the change and solves a flow. The five-row Laplace
table in `GPU/README.md` therefore describes the OLD default and is currently
annotated as such rather than re-run.

This notebook re-runs those five rows. It needs **nvcc only** — `GPU/` is its own
CMake project with no Kokkos, so the build is a couple of minutes rather than the
half hour `colab_gpu_test.ipynb` spends compiling Kokkos.

**Set the runtime to a GPU first**: Runtime > Change runtime type > T4 GPU.

Known from the host build (FP64, 48³, 2000 steps — a different grid, precision
and step count, so NOT a substitute): the Laplace error improved from −19.62 % to
−9.40 % and the spurious current from 1.361e-05 to 1.281e-05. Expect an
improvement; the table wants the actual numbers.

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or
      'no nvidia-smi -- Runtime > Change runtime type > GPU')
cc = subprocess.run(['nvidia-smi','--query-gpu=compute_cap,name','--format=csv,noheader'],
                    capture_output=True, text=True).stdout.strip()
print('device:', cc)
ARCH = cc.split(',')[0].strip().replace('.', '')     # '7.5' -> '75'
print('LBM_GPU_ARCH =', ARCH)
import os; os.environ['ARCH'] = ARCH
# The README table was taken on a T4. A different card is still a valid
# measurement, but say so in the table rather than letting it pass as a T4 row.
if '7.5' not in cc:
    print('\nNOTE: this is NOT a T4. Record the actual device with the numbers.')

## Get the code

The `phase-field-cm-default` branch is pushed, and an anonymous HTTPS clone of it
was verified to work and to carry `MultiOp fluid_op_ = MultiOp::CentralMoments`
— so `MODE = 'clone'` needs nothing uploaded.

`'upload'` remains as the fallback, taking the `m3lb-gpu.tgz` built alongside
this notebook. Whichever route you take, the cell asserts the flipped default is
actually present before spending a build on it: a clone of `main` would
re-measure the OLD numbers and look like a successful run.

In [ ]:
MODE = 'clone'         # 'clone' | 'upload'
BRANCH = 'phase-field-cm-default'
REPO_URL = 'https://github.com/alexderosis/M3LB.git'

import os, glob, shutil, subprocess
SRC = '/content/M3LB'
shutil.rmtree(SRC, ignore_errors=True)

if MODE == 'upload':
    from google.colab import files
    up = files.upload()                       # choose m3lb-gpu.tgz
    name = list(up)[0]
    os.makedirs(SRC, exist_ok=True)
    subprocess.run(['tar','xf',name,'-C',SRC], check=True)
else:
    subprocess.run(['git','clone','--depth','1','-b',BRANCH,REPO_URL,SRC], check=True)

assert os.path.exists(f'{SRC}/GPU/CMakeLists.txt'), 'GPU/CMakeLists.txt not found'

# CHECK THE DEFAULT IS ACTUALLY THE NEW ONE, before spending a build on it.
hdr = open(f'{SRC}/GPU/include/lbm/phasefield.cuh').read()
assert 'MultiOp fluid_op_ = MultiOp::CentralMoments;' in hdr, (
    'this tree still has the BGK fluid default -- you would be re-measuring the '
    'old numbers. Push the branch, or use MODE = upload.')
print('code at', SRC, '-- fluid default is CentralMoments, as intended')

## Build

`bubble` only. nvcc, Release, no Kokkos anywhere in this project.

In [ ]:
%%bash
set -e
cd /content/M3LB/GPU
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DLBM_GPU_ARCH=$ARCH > /dev/null
time cmake --build build -j$(nproc) --target bubble
ls -la build/bubble

## The five rows

64³, R = 16, W = 4, FP32, 40000 steps — the table's own settings. `-nu 0.05`
sets `mu_L = 0.05` and `mu_H = 0.05 * gamma`, i.e. matched KINEMATIC viscosity;
without it `mu_L = mu_H = 0.05`, i.e. matched DYNAMIC viscosity.

The gamma = 100 dynamic-matched row diverged under BGK, and the README argues
that was the viscosity choice and not the model (`omega = 1.994` in the heavy
phase against a limit of 2). Central moments relax the non-hydrodynamic modes at
1 rather than at omega, so it may now survive — worth finding out rather than
assuming either way. It is run last so a divergence does not cost the others.

In [ ]:
%%bash
set -e
cd /content/M3LB/GPU
run () {
  echo "================ $* ================"
  ./build/bubble -n 64 -steps 40000 "$@" 2>&1 | tail -8
  echo
}
run -gamma 1
run -gamma 10
run -gamma 10  -nu 0.05
run -gamma 100 -nu 0.05
run -gamma 100          # the one that diverged under BGK

## The old table, for the diff

These are the rows to replace in `GPU/README.md`, measured on a T4 at 64³, FP32,
40000 steps under the **BGK** fluid default:

| gamma | viscosity | sigma measured | error | spurious current |
|---|---|---|---|---|
| 1 | mu matched | 9.382817e-04 | −6.17% | 1.235e-05 |
| 10 | mu matched | 9.573443e-04 | −4.27% | 1.162e-05 |
| 100 | mu matched | — | **diverged** | — |
| 10 | nu matched | 9.569894e-04 | −4.30% | 2.249e-06 |
| 100 | nu matched | 9.625386e-04 | −3.75% | 5.685e-07 |

Paste the new output back and I will rewrite the table and drop the
"THESE ROWS WERE MEASURED ON THE BGK FLUID" annotation.